Source

https://data.jrc.ec.europa.eu/dataset/9436ea8e-c484-44b8-88f9-810bb6da64ae

In [1]:
import pandas as pd
df = pd.read_csv("STRATEGIC_TRADE_ATLAS_DATA_2026.csv", sep=",")
df.head(6)

,COMMODITY CODE,COMMODITY DESCRIPTION,FLOW,PARTNER,PARTNER ISO2,PARTNER ISO3,PERIOD,REPORTER,REPORTER ISO2,REPORTER ISO3,STRATEGIC COMMODITY SHORT DESCRIPTION,QUANTITY KG,VALUE USD
0,9015.80,"Instruments and appliances used in geodesy, to...",Import,Romania,RO,ROM,12/31/2024,Sweden,SE,SWE,Gravity meters,15,60
1,6903.90,"Retorts, crucibles, mufflers, nozzles, plugs, ...",Import,Spain,ES,ESP,12/31/2024,Egypt,EG,EGY,Ceramic crucibles,112356,411433
2,8481.80,"Appliances for pipes, boiler shells, tanks, va...",Import,Morocco,MA,MAR,12/31/2024,Armenia,AM,ARM,Valves,0,80
3,8421.19,"Centrifuges, incl. centrifugal dryers (excl. i...",Import,Italy,IT,ITA,12/31/2023,TFYR of Macedonia,MK,MKD,Centrifugal separators,180,17673
4,7228.60,Bars and rods of alloy steel other than stainl...,Export,Namibia,NaN,NAM,12/31/2023,United Kingdom,GB,GBR,Specialty steels,40,806
5,8456.30,Machine tools for working any material by remo...,Import,Israel,IL,ISR,12/31/2023,Namibia,NaN,NAM,Electrical Discharge Machines,756,116211


In [4]:
import os

TEST = False

ROWS_PER_FILE = 100000  # data rows per result TTL file
OUTPUT_DIR = "rdf"

os.makedirs(OUTPUT_DIR, exist_ok=True)

file_idx = 0
rows_in_file = 0

def open_output_file(idx):
    f = open(f"{OUTPUT_DIR}/rdf_{idx:04d}.ttl", "w")

    # Namespaces
    f.write("@prefix : <https://purl.org/supply-network/examples/> .\n")
    f.write("@prefix sta: <https://data.jrc.ec.europa.eu/dataset/9436ea8e-c484-44b8-88f9-810bb6da64ae#> .\n")
    f.write("@prefix sn: <https://purl.org/supply-network/onto#> .\n")
    f.write("@prefix cur: <http://qudt.org/vocab/currency/> .\n")
    f.write("@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .\n")
    f.write("\n")

    return f

f = open_output_file(file_idx)

headings_df = df.head(5).copy()
input_regions = headings_df.iloc[1].to_numpy()
input_industries = headings_df.iloc[3].to_numpy()

# Iterate rows
i = 0
for row in df.itertuples(index=False, name=None):

    # Split after ROWS_PER_FILE data rows
    if rows_in_file == ROWS_PER_FILE:
        print(i)
        f.close()
        file_idx += 1
        rows_in_file = 0
        f = open_output_file(file_idx)

    # Read flow direction
    frow_direction = row[2]
    
    # Read countries
    if frow_direction == "Import":
        output_country = row[5]
        input_country =  row[9]
    elif frow_direction == "Export":
        output_country = row[9]
        input_country =  row[5]

    # Read volume
    time = row[6]
    quantity = row[12]
    subject = row[10]

    # Write triples of one supply flow
    supplyflow = f"sta:{i}"
    volume = f"_:{i}"
    f.write(f"{supplyflow} rdf:type sn:SupplyFlow .\n")
    f.write(f"{supplyflow} sn:abstraction :CountryAbstraction .\n")
    f.write(f"{supplyflow} :export sta:{input_country} .\n")
    f.write(f"{supplyflow} :import sta:{output_country} .\n")
    f.write(f"{supplyflow} sn:volume {volume} .\n")
    f.write(f"{volume} rdf:type sn:Volume .\n")
    f.write(f'{volume} sn:time "{time}" .\n')
    f.write(f"{volume} sn:unit cur:USD .\n")
    f.write(f"{volume} sn:quantity {quantity} .\n")
    f.write(f'{volume} sn:subject "{subject}" .\n')

    rows_in_file += 1

    # Only process first two data rows for testing
    if TEST and i >= 2:
        break
    i += 1
    
f.close()

100000
200000
300000
400000
500000
600000
700000
800000
900000
1000000
1100000
1200000
1300000
1400000
1500000
1600000
1700000
1800000
1900000
2000000
2100000
2200000
2300000
2400000
2500000
2600000
2700000
